# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer (FAIR^2) Exploration with `mlcroissant`
This notebook provides a guided, step-by-step template for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the [`mlcroissant`](https://mlcommons.org/croissant/) library. All schema entities (record sets, fields, columns) are referenced by their `@id` in accordance with Croissant best practices.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Show key info from the metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s, as defined in the Croissant schema.

In [ ]:
# List all record sets, their @id, and fields with their @id
from collections import defaultdict

recordset_ids = []
fields_by_recordset = defaultdict(list)

for rs in dataset.record_sets:
    recordset_ids.append(rs['@id'])
    # Each field has an '@id' and 'name'
    if 'field' in rs:
        # Some recordsets may have a single field as dict, or a list
        fields = rs['field']
        if isinstance(fields, dict):
            fields = [fields]
        for f in fields:
            fields_by_recordset[rs['@id']].append({'@id': f['@id'], 'name': f.get('name', f['@id'])})
    print(f"RecordSet: {rs['@id']} | Name: {rs.get('name', '<none>')}")
    if rs['@id'] in fields_by_recordset:
        print("  Fields:")
        for fld in fields_by_recordset[rs['@id']]:
            print(f"    {fld['@id']}: {fld['name']}")

## 3. Data Extraction
Extract the records for each record set using their `@id`, and load them into pandas DataFrames. All entities are referenced by their `@id`.

In [ ]:
# Use the record set IDs found above
dataframes = {}
for record_set_id in recordset_ids:
    rec_gen = dataset.records(record_set=record_set_id)
    # Some record sets may have no rows, so we convert to list
    records = list(rec_gen)
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nLoaded RecordSet {record_set_id}: {df.shape[0]} rows x {df.shape[1]} columns")
        print(f"Columns: {list(df.columns)}")
        display(df.head())
    else:
        print(f"\nRecordSet {record_set_id} has no records.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing and simple transformations to analyze a numeric variable. We 
- filter records by a threshold,
- normalize the values,
- and optionally group by a categorical variable.

Below, choose a record set and field (by `@id`) based on the previous data overview.

In [ ]:
# Example: Use the main clinical data record set
# Pick the first non-empty dataframe as main (you may select by a known @id)

main_rs_id = None
for k, v in dataframes.items():
    if len(v) > 0:
        main_rs_id = k
        break
if not main_rs_id:
    raise ValueError('No non-empty record set found!')
df = dataframes[main_rs_id]
print(f"Main record set: {main_rs_id}")
print(df.dtypes)

# Attempt to select a numeric field by dtype
numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if not numeric_cols:
    # Try to convert likely numeric columns
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

if numeric_cols:
    numeric_field_id = numeric_cols[0]
    print(f"Using numeric field: {numeric_field_id}")
else:
    raise Exception("No numeric fields available for analysis.")

# Filtering: Use a threshold value, e.g., median
threshold = df[numeric_field_id].median()
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalization: Z-score normalization
filtered_df[numeric_field_id + '_normalized'] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())
    / filtered_df[numeric_field_id].std()
)
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

# Grouping: Try grouping by a non-numeric field
cat_cols = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col])]
group_field = None
if cat_cols:
    group_field = cat_cols[0]
    grouped = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Mean {numeric_field_id} by {group_field} (from filtered records):")
    display(grouped.head())

## 5. Visualization
Visualize numeric field distributions and relationships between fields. Here, we plot histograms and bar charts directly from the DataFrame using matplotlib and seaborn where available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
plt.title(f"Histogram of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# If grouping was done, barplot summary
if group_field is not None and 'grouped' in locals():
    plt.figure(figsize=(8,4))
    sns.barplot(data=grouped, x=group_field, y=numeric_field_id)
    plt.title(f"Mean {numeric_field_id} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR^2 dataset using the `mlcroissant` API, referencing all record sets and fields by their Croissant `@id`. After a schema-aware extraction, we performed simple data normalization, filtering, basic grouping, and visualization to help uncover patterns in the clinical dataset.

**Key steps:**
- Schema access and review by `@id`
- Data extraction for each record set
- Numerical analysis and transformation for one field
- Visualization of main results

You can build additional analyses by choosing other record set IDs and field `@id`s per the schema overview above.